# 策略概述

::: {.callout-tip}

投影片操作：**Alt + 點擊** 可縮放任何圖片/表格（reveal.js zoom）；`O` 鍵總覽、`F` 全螢幕。

:::

本策略是三層架構的組合，每一層都經過獨立的單變因 A/B 驗證：

| 層 | 機制 | 單獨貢獻（Top1/SL0 年化） |
| :--- | :--- | :--- |
| **形成期** | Agglomerative 混合特徵分群（價格 PCA ⊕ FMP PIT 基本面 ⊕ GICS） | 1.77%（基底） |
| **進場時機** | 低分散度閘門 DG25：市場橫斷面分散度處於歷史最低四分位時暫停新開倉 | +0.35pp |
| **進場門檻** | entry_z 2.5（高門檻，只交易深度偏離） | +0.33pp |

**組合結果（Top 1，SL 0%）：年化 2.42%｜Sharpe 0.37｜MDD −21.0%｜Profit Factor 1.40**——
報酬、風險調整報酬、回撤、獲利因子四項指標同時優於任一單層版本，且兩個交易端增益近似可加。

實作：形成期 `strategies/formation/agglomerative_FMP.py`；
交易期 `strategies/trading/zscore_trading.py`（`entry_gate` + `entry_z`）；
閘門建構 `run_trading._build_dispersion_gate()`。
config 條目：`Agglomerative Fundamentals (FMP) DG25 EZ`。


## 設計哲學：為什麼是這三層

本策略是一輪系統性消融戰役的產物——六個候選機制中**五個被 A/B 否決**
（動態資金集中、因子殘差化移植、時間停損、提早出場、DRL 時間選單），只有這三層存活：

- 失敗的機制都在「**放大或重組既有訊號**」——幾何複利下集中放大波動拖累、
  殘差表徵與交易空間錯位、時間停損砍在均值回歸的損益谷底
- 存活的機制都在「**選擇何時、何價交易**」——不動任何已開倉位的生命週期，
  只在事前可觀測的狀態下決定要不要進場、多深才進場

負面結果的完整診斷保留於 `archive/config_archived_strategies.py`「交易端微觀規則戰役」節。


# 參考文獻與引用對應


## 文獻 1：Hong & Hwang (2021)

> Hong, S., & Hwang, S. (2021). In search of pairs using firm fundamentals: Is pairs trading profitable? *The European Journal of Finance*, **29**(5).

**參考部分**：以企業基本面特徵識別配對——基本面相似的公司共享現金流與估值驅動因子，價格間長期均衡有經濟基礎。

**為何參考**：形成期基本面區塊（log 市值、盈餘殖利率 $1/PE$）的直接依據；
本策略用 **FMP Point-in-Time 逐點資料**（每個形成窗只取當時可得的基本面），修正了免費快照資料的前視偏誤。

📄 `ref/2021-In Search of Pairs using Firm Fundamentals.pdf`


## 文獻 2：Gatev, Goetzmann & Rouwenhorst (2006)

> Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading: Performance of a relative value arbitrage rule. *Review of Financial Studies*, **19**(3), 797–827.

**參考部分**：

- Z-Score 進出場框架與形成期統計量凍結設計（交易層基礎）
- **獲利的 regime 相依性**：其報酬分解顯示配對交易利潤在市場動盪期顯著較高——
  套利者因在混亂時執行一價定律而獲得風險溢酬，平靜時期無溢酬可領

**為何參考**：

- 低分散度閘門（DG25）正是把此觀察轉為**事前規則**：本策略實測日損益依橫斷面分散度
  四分位分層——Q4（最高 25%）貢獻 +8,674、Q1+Q2 合計 −4,920，
  **全部淨利來自分散度最高的四分之一交易日**；閘門在無溢酬的平靜期停止支付交易成本

📄 `ref/2006-Pairs Trading Performance of a Relative-Value Arbitrage Rule.pdf`


## 文獻 3：Do & Faff (2012)

> Do, B., & Faff, R. (2012). Are pairs trading profits robust to trading costs? *Journal of Financial Research*, **35**(2), 261–287.

**參考部分**：交易成本（單邊約 30 bps）對配對交易獲利的侵蝕；淺偏離交易在計入成本後多為淨虧損。

**為何參考**：

- 本策略成本假設（單邊 0.29%，往返 0.58%）之出處
- **entry_z 2.5 高門檻的經濟邏輯**：預期擷取 $\approx entry_z \times \sigma_{spread}$，
  門檻越高、單筆預期擷取對往返成本的倍數越大——放棄淺偏離的頻繁小交易，
  只交易深度偏離（基準 EZ 掃描：2.0 → 2.5 → 3.0 的 PF 從 1.20 → 1.29 → 1.35 單調上升）

📄 `ref/2012 - ARE PAIRS TRADING PROFITS ROBUST TO TRADING COSTS.pdf`


## 文獻 4：Ward (1963)／Avellaneda & Lee (2010)／Krauss et al. (2016)

> Ward, J. H. (1963). Hierarchical grouping to optimize an objective function. *JASA*, **58**(301).
> Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7).
> Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies. *EJOR*.

**參考部分與理由**（形成期元件，詳見 `formation/agglomerative_fundamentals.ipynb`）：

- Ward：階層聚類框架與 dendrogram 分位數校準 `distance_threshold`（⚠️ `ref/` 缺 PDF，需補充）
- Avellaneda & Lee：報酬 PCA 因子載荷＝價格行為區塊（📄 `ref/2010-Statistical arbitrage in the U.S. equities market.pdf`）
- Krauss et al.：ADF／半衰期／Hurst 三道統計過濾（📄 `ref/2016-...pdf`）


# 各階段行為

## 架構總覽

```
形成期（252d，每 21d 滾動）                交易期（126d）
┌─────────────────────────────┐   ┌──────────────────────────────┐
│ 1. FMP PIT 基本面（≤形成期末） │   │ 4. 低分散度閘門（walk-forward）│
│ 2. 混合特徵 Agglomerative 分群 │ → │ 5. Z-Score 狀態機（EZ 2.5）    │
│ 3. 群內 min-SSD + 三道統計過濾 │   │ 6. 風控（SL 網格／期末強平）    │
└─────────────────────────────┘   └──────────────────────────────┘
```


## 階段 1–3：形成期（同 Agglomerative FMP，摘要）

1. **PIT 基本面**：從 FMP Parquet 取「日期 ≤ 形成期末」的最近一筆市值／本益比／產業——每窗只用當時可得資訊，無前視
2. **混合特徵分群**：價格 PCA 因子載荷（5 維）⊕ $[\log(1+\text{MktCap}), 1/PE]$ ⊕ GICS one-hot，
   三區塊各自標準化加權拼接 → Agglomerative（average linkage，合併距離 75 分位校準門檻）
3. **群內配對**：min-SSD 排序 + ADF（$p<0.05$）＋半衰期 $[1,42]$ 日＋Hurst $<0.5$，取前 `top_n`

完整公式見 `formation/agglomerative_fundamentals.ipynb`（FMP 資料源變體）。


## 階段 4：低分散度閘門（DG25，本策略核心創新）

**訊號建構**（`_build_dispersion_gate`，全程 walk-forward）：

$$d_t = \text{std}_{\text{橫斷面}}\big(r_{t,1}, \dots, r_{t,N}\big), \qquad
D_t = \frac{1}{30}\sum_{s=t-30}^{t-1} d_s \quad (\text{shift(1)：只用昨日以前})$$

$$\text{pct}_t = \text{expanding rank}(D_t \mid D_{252:t}), \qquad
\text{gate}_t = \big[\, \text{pct}_t \ge 0.25 \,\big]$$

**規則**：$\text{gate}_t = \text{False}$（分散度處於歷史最低四分位）的日子**暫停新開倉**；
已持倉位的評價、出場、停損完全不受影響。expanding 分位需 252 日暖身，暖身期一律放行。

**為何有效**（機制）：

1. **偏離的供給**：高分散度期個股受異質衝擊大量錯位，真實偏離供給充足；
   低分散度動能市中觸發 $|Z|>entry_z$ 的多為噪音或單邊趨勢的開端
2. **溢酬的時變性**：套利溢酬只在市場混亂時存在（Gatev 2006），平靜期進場是支付 0.58% 成本換取不存在的溢酬
3. **結構安全性**：不碰已開倉位的生命週期（先前時間停損／提早出場的失敗教訓），
   只依事前可觀測狀態決定「今天要不要開新倉」


## 階段 5：Z-Score 狀態機（entry_z 2.5）

$$Z_t = \text{clip}\left(\frac{P'_{A,t} - \beta P'_{B,t} - \mu_\epsilon^{form}}{\sigma_\epsilon^{form}},\ -10,\ 10\right)$$

| 條件 | 動作 |
| :--- | :--- |
| $\text{gate}_t$ 允許 且 $Z_t > 2.5$ | 空 A、多 B（風險中性配置 $v_A = C/W$、$v_B = |\beta|C/W$） |
| $\text{gate}_t$ 允許 且 $Z_t < -2.5$ | 多 A、空 B |
| $\text{gate}_t$ 禁止 且 $|Z_t| > 2.5$ | `HOLD_CASH (LOW_DISP_GATE)`——訊號成立但不進場 |
| 空頭且 $Z_t \le 0$／多頭且 $Z_t \ge 0$ | 平倉（可再進場） |
| 期末仍持倉 | 強制結算 |

**高門檻的取捨**：交易數 733 → 589（−20%），但單筆品質提升（PF 1.20 → 1.40）——
放棄「多而淺」換「少而深」，每筆預期擷取對 0.58% 往返成本的覆蓋倍數更高。


## 階段 6：風控與會計

與標準 Z-Score 引擎完全相同：比例停損（網格）、期末強平、進出場各扣 0.29% 摩擦、
每配對獨立資金 $10{,}000、`Daily_Delta` 逐日會計。詳見 `trading/zscore_trading.ipynb`。


# 消融驗證：每一層都被單獨檢驗過

四方對照（Top 1，SL 0%，同一組 FMP 形成期配對）：

| 版本 | 年化 | Sharpe | MDD | PF | 交易數 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| 基準（EZ2.0，無閘門） | 1.77% | 0.26 | −24.7% | 1.20 | 733 |
| 只加高門檻 EZ2.5 | 2.10% | 0.31 | −21.9% | 1.29 | 708 |
| 只加閘門 DG25 | 2.12% | 0.31 | −23.9% | 1.30 | 537 |
| **DG25 × EZ2.5（本策略）** | **2.42%** | **0.37** | **−21.0%** | **1.40** | 589 |

- 兩個增益近似可加（+0.33pp 與 +0.35pp → 組合 +0.65pp），顯示兩機制作用於**不同的虧損來源**
  （淺偏離噪音交易 vs 平靜期無溢酬交易），非同一效應的重複計算
- DG25 在其餘 TopN 亦全格改善（含 MDD 下降），非單格挑選——見 `comparison.ipynb`


# 延伸：閘門 × DRL 門檻選擇式

同一閘門疊加於 DRL-THR 交易端（`Agglomerative Fundamentals DRL THR (FMP) DG25`）。
閘門同時作用於**反事實標籤生成**與正式模擬——agent 學到的是「閘門世界裡」各門檻的
可實現報酬，兩機制在訓練層即協同。

各 TopN 最佳停損檔對照（單次訓練）：

| TopN | DRL 基準 | DRL × DG25 |
| :--- | :--- | :--- |
| Top 1 | 2.25%／Sh 0.33 | **2.68%／Sh 0.40**（PF 1.47，MDD −21.9%） |
| Top 3 | 0.16% | 0.59% |
| Top 5 | 0.23% | 0.44% |
| Top 10 | −0.43% | 0.17%（轉正） |
| Top 20 | −0.11% | 0.06%（轉正） |

- **15/15 網格全數正 Sharpe**（基準 8/15）——本研究首個全網格為正的參數面
- DRL-THR 未固定隨機種子，上表為單次訓練值；多輪重跑的中位數±範圍
  （`tools/run_drl_variance.py`，5 輪）為正式引用口徑，見 `comparison.ipynb` 變異數章節


# 參數總表

| 參數 | 值 | 說明 |
| :--- | :---: | :--- |
| 形成窗 / 交易窗 / 滾動 | 252 / 126 / 21 交易日 | 標準滾動回測 |
| `top_n` | 網格 [1,3,5,10,20]（旗艦 Top 1） | 每期配對數 |
| 形成期 | 同 Agglomerative (FMP) | PIT 基本面分群 |
| `disp_gate_pctl` | **25** | 分散度歷史分位 < 25% 暫停新開倉 |
| 閘門訊號 | 30 日均分散度，shift(1)，expanding 分位（252d 暖身） | walk-forward 無前視 |
| `entry_z` / `exit_z` | **2.5** / 0.0 | 高門檻進場、回歸均值出場 |
| `stop_loss_pct` | 0（旗艦格） | 比例停損不啟用 |
| `fee_rate` | 0.0029 單邊 | Do & Faff (2012) |

## 已知限制

- 閘門依賴的分散度分位在極端 regime 轉折（如 2020/03）有約一個月的辨識落後
- 形成期繼承 Agglomerative FMP 的全部限制（PIT 月頻粒度、部分已下市股票基本面缺失以中位數插補）
- 上表為單一參數組的全樣本回測結果；跨參數穩健性見 `comparison.ipynb` 全網格分析
